![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/4.15.HL7v2_DeIdentification.ipynb)

## 🔐 Spark Transformer for De-identifying HL7 v2.x Messages

This Spark-based transformer de-identifies **HL7 v2.x** messages — the classic pipe / hat delimited "vertical bar" format (e.g. `ADT^A01`, `ORU^R01`, `RDE^O01`) — according to configurable privacy rules.

It is the newest member of the **De-identification family**, a sibling to `FhirDeIdentification` and `CdaDeIdentification`, and shares the same obfuscation engine (`mappingRules`, date-shifting, `deidentify` / `deidentify_list` / `transform`).

---

## ✅ Key Features

- 📦 **Terser-style path rules**: Target any field with `SEG-field-component-subcomponent` notation (e.g. `PID-5-1`, `MSH-7`)
- 🌐 **Version-agnostic**: A single lightweight, dependency-free parser handles v2.1 → v2.9 uniformly
- 🧩 **Segment occurrences & repetitions**: `NK1(2)-2-1` targets the 2nd `NK1`; a whole field obfuscates every `~` repetition
- 🧠 **Structured field obfuscation**: Names, dates, addresses, phone numbers, SSNs, medical-record / account / device identifiers, provider names
- 📝 **Free-text de-identification**: Narrative fields (`OBX-5`, `NTE-3`) processed with NLP pipelines
- 🔁 **Robust to escaped exports**: Handles both real `\r` separators and backslash-escaped exports (literal `\r`, doubled `MSH-2`), preserving the input's style on output
- ⚙️ **Modes**: `mask` or `obfuscate`

---

## 📦 The Two Pretrained Models

Healthcare NLP 7.0.0 ships two ready-to-use `Hl7v2DeIdentification` models. Both bake in the same comprehensive default PHI map (280 Terser paths) covering the HIPAA Safe Harbor identifiers: names, dates, addresses, phones, SSNs, MRN / account / member / license / device IDs and clinician names. The map spans the common HL7 v2 segments (`MSH`, `EVN`, `PID`, `PD1`, `MRG`, `NK1`, `PV1`, `PV2`, `GT1`, `IN1`/`IN2`, `AL1`, `DG1`, `PR1`, `ROL`, `ACC`, `UB1`/`UB2`, `TXA`, `SCH`/`AIP`/`AIS`, `ORC`, `OBR`, `OBX`, `SPM`, `RXA`, `RXE`). Segments that are absent from a message are skipped, so the same model works on any HL7 v2 feed.

| Model | Annotator | What it de-identifies | Free-text paths | NLP pipeline |
| --- | --- | --- | --- | --- |
| `hl7v2_deidentification_base` | `Hl7v2DeIdentification` | Structured PHI fields only | none | not needed |
| `hl7v2_deidentification_free_text` | `Hl7v2DeIdentification` | Structured PHI fields **and** the narrative fields `OBX-5` and `NTE-3` | `OBX-5`, `NTE-3` | required (`setPipeline()`) |

- **`hl7v2_deidentification_base`** de-identifies the structured PHI fields of an HL7 v2 message with the built-in PHI map. Narrative fields such as `OBX-5` and `NTE-3` are left as they are, so it is the right choice when your feed carries no free text or when free text is handled elsewhere.
- **`hl7v2_deidentification_free_text`** applies the same structured map and additionally routes `OBX-5` (observation value) and `NTE-3` (comment) through a Healthcare NLP de-identification pipeline, so PHI written in prose is de-identified too. It needs a pipeline, which you attach with `setPipeline()`.

Both models are small (about 17 KB): they hold the path map and defaults, and the free-text model relies on the pipeline you give it for the narrative. Load them with `Hl7v2DeIdentification.pretrained("<model name>", "en", "clinical/models")` and use the same parameters, listed below.

| **Parameter**                   | **Description**                                                                                                                                                                                                           |
| ------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **mode**                        | Mode for Anonymizer. Options: [`mask`, `obfuscate`]                                                                                                                                                                       |
| **mappingRules**                | Dictionary mapping HL7 v2 Terser paths to entity types (e.g. `{"PID-5-1": "LAST_NAME", "PID-7": "DATE"}`). Components/subcomponents are 1-based; `MSH` uses real HL7 field numbers (`MSH-7` = message date/time).         |
| **freeTextPaths**               | List of HL7 v2 paths to free-text fields (e.g. `["OBX-5", "NTE-3"]`).<br><br>These paths are processed using the de-identification pipeline set via `setPipeline()`.                                                       |
| **pipeline**                    | Spark NLP Pipeline object (`PipelineModel` or `PretrainedPipeline`) for free-text sections.<br><br>Set via `setPipeline(spark, pipeline, outputCol="obfuscated")`, where `outputCol` is the pipeline's de-identified text column (default: `"obfuscated"`). |
| **dateEntities**                | List of date entities. Default: [`DATE`, `DOB`, `DOD`]                                                                                                                                                                    |
| **unnormalizedDateMode**        | Mode to use for unformatted dates. Options: [`mask`, `obfuscate`, `skip`]. Default: `obfuscate`                                                                                                                           |
| **days**                        | Number of days to displace dates for obfuscation. If not set, a random value between 1 and 60 is used. Dates are *shifted*, not replaced — set `days` (non-zero) to see them change.                                       |
| **dateFormats**                 | List of date formats to displace if parsed. Default for HL7 is `yyyyMMdd` (the `HHmmss` remainder of a timestamp is preserved).                                                                                            |
| **obfuscateRefSource**          | Source for obfuscating entities (non-date). Options: [`custom`, `faker`, `both`]                                                                                                                                          |
| **language**                    | Language for regex and faker data. Options: [`en`, `de`, `es`, `fr`, `ar`, `ro`]. Default: `en`                                                                                                                           |
| **seed**                        | Seed for deterministic obfuscation results.                                                                                                                                                                               |
| **maskingPolicy**               | Masking policy. Options: [`same_length_chars`, `entity_labels`, `fixed_length_chars`]                                                                                                                                     |
| **fixedMaskLength**             | Mask length when using `fixed_length_chars` policy.                                                                                                                                                                       |
| **sameLengthFormattedEntities** | Formatted entities to preserve length during obfuscation. Supported: PHONE, FAX, ID, IDNUM, BIOID, MEDICALRECORD, ZIP, VIN, SSN, DLN, LICENSE, PLATE                                                                       |
| **customFakers**                | Custom dictionary of faker terms to use for specific entities.                                                                                                                                                            |
| **keepYear** / **keepMonth**    | Keep year / month intact during date obfuscation. Default: False                                                                                                                                                          |

# Setup

In [ ]:
import json, os
from google.colab import files

if 'spark_jsl.json' not in os.listdir():
  license_keys = files.upload()
  os.rename(list(license_keys.keys())[0], 'spark_jsl.json')

with open('spark_jsl.json') as f:
    license_keys = json.load(f)

# Defining license key-value pairs as local variables
locals().update(license_keys)
os.environ.update(license_keys)

In [ ]:
# Installing pyspark and spark-nlp
! pip install --upgrade -q pyspark==4.0.1 spark-nlp==$PUBLIC_VERSION

# Installing Spark NLP Healthcare
! pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION  --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Installing Spark NLP Display Library for visualization
! pip install -q spark-nlp-display


In [3]:
from pyspark.ml import Pipeline,PipelineModel
import sparknlp_jsl
import sparknlp
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp.base import *
from sparknlp.pretrained import PretrainedPipeline
import warnings
warnings.filterwarnings('ignore')
from IPython.display import clear_output

spark = sparknlp_jsl.start(secret = license_keys["SECRET"])

print ("Spark NLP Version :", sparknlp.version())
print ("Spark NLP_JSL Version :", sparknlp_jsl.version())

spark

Spark NLP Version : 7.0.0
Spark NLP_JSL Version : 7.0.0


# Util Functions

In [4]:
def show_hl7_diff(original: str, deidentified: str):
    """Show only the fields that changed between two HL7 v2 messages (segments in the same order)"""
    normalize = lambda m: m.replace("\r\n", "\n").replace("\r", "\n").strip().split("\n")
    for o_segment, d_segment in zip(normalize(original), normalize(deidentified)):
        o_fields, d_fields = o_segment.split("|"), d_segment.split("|")
        segment = o_fields[0]
        for i, (o, d) in enumerate(zip(o_fields, d_fields)):
            if o != d:
                field = f"{segment}-{i + 1}" if segment == "MSH" else f"{segment}-{i}"
                print(f"Changed: {field} | {o} → {d}")

# Load the Pretrained Models

The free-text model hands `OBX-5` / `NTE-3` to a de-identification pipeline, so we load one first. `clinical_deidentification_docwise_benchmark_optimized_v2` returns its de-identified text in the `obfuscated` column, which is the column name we pass to `setPipeline()`.

In [5]:
deid_pipeline = PretrainedPipeline("clinical_deidentification_docwise_benchmark_optimized_v2", "en", "clinical/models")

clinical_deidentification_docwise_benchmark_optimized_v2 download started this may take some time.
Approx size to download 1.7 GB
[OK!]


We load each model **once** and keep it as it is. The pretrained models ship with their own `days` and `seed` values, so we set both explicitly to get the same output on every run.

> **Tip:** to try a different configuration (custom rules, another mode), call `.copy({})` on the loaded model and change the copy. A copy does not carry the pipeline, so hand the same `deid_pipeline` variable to `setPipeline()` on a copy of the free-text model. Calling `setMappingRules()` directly on a model loaded with `pretrained()` also changes what a later `pretrained()` call with the same name returns in this session.

In [6]:
hl7_base = (
    Hl7v2DeIdentification.pretrained("hl7v2_deidentification_base", "en", "clinical/models")
      .setInputCol("text")
      .setOutputCol("deid")
      .setMode("obfuscate")
      .setDays(20)
      .setSeed(88)
)

hl7v2_deidentification_base download started this may take some time.
Approximate size to download 17.1 KB
[OK!]


In [7]:
hl7_free_text = (
    Hl7v2DeIdentification.pretrained("hl7v2_deidentification_free_text", "en", "clinical/models")
      .setInputCol("text")
      .setOutputCol("deid")
      .setMode("obfuscate")
      .setDays(20)
      .setSeed(88)
      .setPipeline(spark, deid_pipeline, "obfuscated")
)

hl7v2_deidentification_free_text download started this may take some time.
Approximate size to download 17.1 KB
[OK!]


## 🗺️ What Is Inside the Models: the Default PHI Map

Both models carry the default PHI map as their `mappingRules`. The free-text model also carries the free-text paths.

In [8]:
default_rules = dict(hl7_base.getMappingRules())   # keep our own copy of the default map

print("hl7_base       mappingRules :", len(hl7_base.getMappingRules()))
print("hl7_free_text  mappingRules :", len(hl7_free_text.getMappingRules()))
print("hl7_free_text  freeTextPaths:", hl7_free_text.getFreeTextPaths())
print("Same structured map in both :", dict(hl7_base.getMappingRules()) == dict(hl7_free_text.getMappingRules()))

hl7_base       mappingRules : 280
hl7_free_text  mappingRules : 280
hl7_free_text  freeTextPaths: ['OBX-5', 'NTE-3']
Same structured map in both : True


In [9]:
# A few entries of the default map (Terser path -> entity type)
for path in ["MSH-7", "PID-3-1", "PID-5-1", "PID-5-2", "PID-7", "PID-11-1", "PID-13", "PID-19", "PV1-7-2", "IN1-16-1"]:
    print(f"{path:10s} -> {default_rules[path]}")

MSH-7      -> DATE
PID-3-1    -> ID
PID-5-1    -> LAST_NAME
PID-5-2    -> FIRST_NAME
PID-7      -> DATE
PID-11-1   -> STREET
PID-13     -> PHONE
PID-19     -> SSN
PV1-7-2    -> LAST_NAME
IN1-16-1   -> LAST_NAME


# Basic Examples to Show Functionalities

HL7 v2 messages are made of segments (one per line) built from fields (`|`), components (`^`), repetitions (`~`) and subcomponents (`&`). We use two synthetic messages, no real PHI:

- an `ADT^A01` (patient admit) message with structured PHI only,
- an `ORU^R01` (lab result) message that also carries narrative PHI in `OBX-5` and `NTE-3`.

In [10]:
# Synthetic ADT^A01 admit message. Raw string so MSH-2 "^~\&" stays intact.
hl7_adt = r"""MSH|^~\&|GHH_ADT|GOOD HEALTH HOSPITAL|GHH_LAB|GOOD HEALTH LAB|20240108071200||ADT^A01|MSG00001|P|2.5
EVN|A01|20240108071200
PID|1||MRN123456^^^GHH^MR||DOE^JOHN^MICHAEL||19800101|M||W|123 Main Street^^Boston^MA^02108^USA||617-555-1122|617-555-4466|EN|S||ACC998877|123-45-6789
NK1|1|DOE^JANE^K|SPO|123 Main Street^^Boston^MA^02108^USA|617-555-1144
PV1|1|I|WARD1^101^A||||004777^SMITH^ROBERT^A^^^Dr|008064^JONES^MARY^B^^^Dr|||||||||004777^SMITH^ROBERT^A^^^Dr||VN123|||||||||||||||||||||||||20240108071200
IN1|1|PLAN01|INS01|Good Health Insurance||||||||||||DOE^JOHN^MICHAEL|SELF|19800101|123 Main Street^^Boston^MA^02108^USA"""

print(hl7_adt)

MSH|^~\&|GHH_ADT|GOOD HEALTH HOSPITAL|GHH_LAB|GOOD HEALTH LAB|20240108071200||ADT^A01|MSG00001|P|2.5
EVN|A01|20240108071200
PID|1||MRN123456^^^GHH^MR||DOE^JOHN^MICHAEL||19800101|M||W|123 Main Street^^Boston^MA^02108^USA||617-555-1122|617-555-4466|EN|S||ACC998877|123-45-6789
NK1|1|DOE^JANE^K|SPO|123 Main Street^^Boston^MA^02108^USA|617-555-1144
PV1|1|I|WARD1^101^A||||004777^SMITH^ROBERT^A^^^Dr|008064^JONES^MARY^B^^^Dr|||||||||004777^SMITH^ROBERT^A^^^Dr||VN123|||||||||||||||||||||||||20240108071200
IN1|1|PLAN01|INS01|Good Health Insurance||||||||||||DOE^JOHN^MICHAEL|SELF|19800101|123 Main Street^^Boston^MA^02108^USA


In [11]:
# Synthetic ORU^R01 lab result with a narrative OBX-5 and an NTE-3 note.
hl7_oru = r"""MSH|^~\&|LAB|GOOD HEALTH LAB|EHR|GOOD HEALTH HOSPITAL|20240108090000||ORU^R01|MSG00002|P|2.4
PID|1||MRN123456^^^GHH^MR||DOE^JOHN^MICHAEL||19800101|M|||123 Main Street^^Boston^MA^02108^USA||617-555-1122
OBR|1|ORD123|FILL456|CBC^Complete Blood Count|||20240108083000|||||||||004777^SMITH^ROBERT
OBX|1|TX|NARR^Narrative||Patient John Doe was seen on 01/08/2024 at Good Health Hospital and reports mild fatigue. He lives at 123 Main Street, Boston and can be reached at 617-555-1122.||||||F
NTE|1||Follow up with Dr. Robert Smith at 617-555-9000 in two weeks."""

print(hl7_oru)

MSH|^~\&|LAB|GOOD HEALTH LAB|EHR|GOOD HEALTH HOSPITAL|20240108090000||ORU^R01|MSG00002|P|2.4
PID|1||MRN123456^^^GHH^MR||DOE^JOHN^MICHAEL||19800101|M|||123 Main Street^^Boston^MA^02108^USA||617-555-1122
OBR|1|ORD123|FILL456|CBC^Complete Blood Count|||20240108083000|||||||||004777^SMITH^ROBERT
OBX|1|TX|NARR^Narrative||Patient John Doe was seen on 01/08/2024 at Good Health Hospital and reports mild fatigue. He lives at 123 Main Street, Boston and can be reached at 617-555-1122.||||||F
NTE|1||Follow up with Dr. Robert Smith at 617-555-9000 in two weeks.


## 📊 Spark DataFrames

The models are Spark transformers: put messages in a DataFrame column and call `transform()`. The free-text model runs the NLP pipeline for every message, so we process its messages one by one with `deidentify()`, as in the next sections.

In [12]:
hl7_df = spark.createDataFrame([[hl7_adt], [hl7_oru]]).toDF("text")
hl7_base.transform(hl7_df).select("deid").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|deid                                                                                                                                                                                                                                                                                                         

## 🛡️ Direct HL7 De-identification Without Spark DataFrames

`deidentify()` takes a message string and returns the de-identified string. We run the ADT message through both models and list what changed.

In [13]:
adt_base = hl7_base.deidentify(hl7_adt)
print(adt_base)

MSH|^~\&|GHH_ADT|GOOD HEALTH HOSPITAL|GHH_LAB|GOOD HEALTH LAB|20240128071200||ADT^A01|MSG00001|P|2.5
EVN|A01|20240128071200
PID|1||TMC678901^^^GHH^MR||CEASE^MYLENE^ANOLA||19800121|M||W|1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN||162-000-6677|162-000-9911|EN|V||PHH443322|678-90-1234
NK1|1|CEASE^DELTA^FARRELL|SPO|1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN|162-000-6699
PV1|1|I|WARD1^101^A||||559222^LITTER^DERREK^A^^^Dr|553519^PROW^RODERICK^B^^^Dr|||||||||559222^LITTER^DERREK^A^^^Dr||WC678|||||||||||||||||||||||||20240128071200
IN1|1|PLAN01|INS01|Good Health Insurance||||||||||||CEASE^MYLENE^ANOLA|SELF|19800121|1011 North Cooper Street^^Allenchester^VA^57653^USA


In [14]:
adt_free_text = hl7_free_text.deidentify(hl7_adt)
print(adt_free_text)

MSH|^~\&|GHH_ADT|GOOD HEALTH HOSPITAL|GHH_LAB|GOOD HEALTH LAB|20240128071200||ADT^A01|MSG00001|P|2.5
EVN|A01|20240128071200
PID|1||TMC678901^^^GHH^MR||CEASE^MYLENE^ANOLA||19800121|M||W|1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN||162-000-6677|162-000-9911|EN|V||PHH443322|678-90-1234
NK1|1|CEASE^DELTA^FARRELL|SPO|1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN|162-000-6699
PV1|1|I|WARD1^101^A||||559222^LITTER^DERREK^A^^^Dr|553519^PROW^RODERICK^B^^^Dr|||||||||559222^LITTER^DERREK^A^^^Dr||WC678|||||||||||||||||||||||||20240128071200
IN1|1|PLAN01|INS01|Good Health Insurance||||||||||||CEASE^MYLENE^ANOLA|SELF|19800121|1011 North Cooper Street^^Allenchester^VA^57653^USA


In [15]:
print("Identical output on a message without free text:", adt_base == adt_free_text)
print()
show_hl7_diff(hl7_adt, adt_base)

Identical output on a message without free text: True

Changed: MSH-7 | 20240108071200 → 20240128071200
Changed: EVN-2 | 20240108071200 → 20240128071200
Changed: PID-3 | MRN123456^^^GHH^MR → TMC678901^^^GHH^MR
Changed: PID-5 | DOE^JOHN^MICHAEL → CEASE^MYLENE^ANOLA
Changed: PID-7 | 19800101 → 19800121
Changed: PID-11 | 123 Main Street^^Boston^MA^02108^USA → 1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN
Changed: PID-13 | 617-555-1122 → 162-000-6677
Changed: PID-14 | 617-555-4466 → 162-000-9911
Changed: PID-16 | S → V
Changed: PID-18 | ACC998877 → PHH443322
Changed: PID-19 | 123-45-6789 → 678-90-1234
Changed: NK1-2 | DOE^JANE^K → CEASE^DELTA^FARRELL
Changed: NK1-4 | 123 Main Street^^Boston^MA^02108^USA → 1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN
Changed: NK1-5 | 617-555-1144 → 162-000-6699
Changed: PV1-7 | 004777^SMITH^ROBERT^A^^^Dr → 559222^LITTER^DERREK^A^^^Dr
Changed: PV1-8 | 008064^JONES^MARY^B^^^Dr → 553519^PROW^RODERICK^B^^^Dr
Changed: PV1-17 | 004777^SMITH^ROB

## ⚙️ Supports a List of HL7 Messages with Multi-threaded Processing Under the Hood

`deidentify_list()` de-identifies a list of messages in parallel. For the free-text model we loop over the messages, because each message needs its own pipeline run.

In [16]:
deidentified_list_base = hl7_base.deidentify_list([hl7_adt, hl7_oru])

print(type(deidentified_list_base))
print(f"Number of messages (base) : {len(deidentified_list_base)}")

<class 'list'>
Number of messages (base) : 2


In [17]:
deidentified_list_free_text = [hl7_free_text.deidentify(message) for message in [hl7_adt, hl7_oru]]

print(type(deidentified_list_free_text))
print(f"Number of messages (free_text) : {len(deidentified_list_free_text)}")

<class 'list'>
Number of messages (free_text) : 2


## 🧠 Free-Text De-identification with NLP Pipelines

HL7 v2 messages often carry narrative PHI in `OBX-5` (observation value) and `NTE-3` (notes). The `free_text` model sends those two fields through the de-identification pipeline; the `base` model leaves them untouched. Run the ORU message through both:

In [18]:
oru_base = hl7_base.deidentify(hl7_oru)
print(oru_base)

MSH|^~\&|LAB|GOOD HEALTH LAB|EHR|GOOD HEALTH HOSPITAL|20240128090000||ORU^R01|MSG00002|P|2.4
PID|1||TMC678901^^^GHH^MR||CEASE^MYLENE^ANOLA||19800121|M|||1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN||162-000-6677
OBR|1|ORD123|FILL456|CBC^Complete Blood Count|||20240128083000|||||||||559222^LITTER^DERREK
OBX|1|TX|NARR^Narrative||Patient John Doe was seen on 01/08/2024 at Good Health Hospital and reports mild fatigue. He lives at 123 Main Street, Boston and can be reached at 617-555-1122.||||||F
NTE|1||Follow up with Dr. Robert Smith at 617-555-9000 in two weeks.


In [19]:
oru_free_text = hl7_free_text.deidentify(hl7_oru)
print(oru_free_text)

MSH|^~\&|LAB|GOOD HEALTH LAB|EHR|GOOD HEALTH HOSPITAL|20240128090000||ORU^R01|MSG00002|P|2.4
PID|1||TMC678901^^^GHH^MR||CEASE^MYLENE^ANOLA||19800121|M|||1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN||162-000-6677
OBR|1|ORD123|FILL456|CBC^Complete Blood Count|||20240128083000|||||||||559222^LITTER^DERREK
OBX|1|TX|NARR^Narrative||Patient Valerie Ates was seen on 18/09/2024 at Medical Center Of Trinity and reports mild fatigue. He lives at 2001 Ladbrook Drive, Winder and can be reached at 839-777-3344.||||||F
NTE|1||Follow up with Dr. Elidia Plump at 839-777-1222 in two weeks.


Fields that changed in each output. Only the `free_text` model touches `OBX-5` and `NTE-3`:

In [20]:
print("hl7v2_deidentification_base")
show_hl7_diff(hl7_oru, oru_base)

hl7v2_deidentification_base
Changed: MSH-7 | 20240108090000 → 20240128090000
Changed: PID-3 | MRN123456^^^GHH^MR → TMC678901^^^GHH^MR
Changed: PID-5 | DOE^JOHN^MICHAEL → CEASE^MYLENE^ANOLA
Changed: PID-7 | 19800101 → 19800121
Changed: PID-11 | 123 Main Street^^Boston^MA^02108^USA → 1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN
Changed: PID-13 | 617-555-1122 → 162-000-6677
Changed: OBR-7 | 20240108083000 → 20240128083000
Changed: OBR-16 | 004777^SMITH^ROBERT → 559222^LITTER^DERREK


In [21]:
print("hl7v2_deidentification_free_text")
show_hl7_diff(hl7_oru, oru_free_text)

hl7v2_deidentification_free_text
Changed: MSH-7 | 20240108090000 → 20240128090000
Changed: PID-3 | MRN123456^^^GHH^MR → TMC678901^^^GHH^MR
Changed: PID-5 | DOE^JOHN^MICHAEL → CEASE^MYLENE^ANOLA
Changed: PID-7 | 19800101 → 19800121
Changed: PID-11 | 123 Main Street^^Boston^MA^02108^USA → 1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN
Changed: PID-13 | 617-555-1122 → 162-000-6677
Changed: OBR-7 | 20240108083000 → 20240128083000
Changed: OBR-16 | 004777^SMITH^ROBERT → 559222^LITTER^DERREK
Changed: OBX-5 | Patient John Doe was seen on 01/08/2024 at Good Health Hospital and reports mild fatigue. He lives at 123 Main Street, Boston and can be reached at 617-555-1122. → Patient Valerie Ates was seen on 18/09/2024 at Medical Center Of Trinity and reports mild fatigue. He lives at 2001 Ladbrook Drive, Winder and can be reached at 839-777-3344.
Changed: NTE-3 | Follow up with Dr. Robert Smith at 617-555-9000 in two weeks. → Follow up with Dr. Elidia Plump at 839-777-1222 in two weeks.


## 📝 Path Notation Examples

HL7 v2 de-identification uses Terser-style paths (1-based components):

| Path            | Targets                                                          |
| --------------- | --------------------------------------------------------------- |
| `PID-5`         | Field 5 (all components / repetitions), the whole patient name  |
| `PID-5-1`       | Field 5, component 1, the family name                           |
| `PID-5-1-2`     | Field 5, component 1, subcomponent 2                            |
| `PID-13`        | Field 13 (all `~` repetitions), every home phone                |
| `PID(2)-5-1`    | The **2nd** `PID` segment occurrence, family name               |
| `NK1(2)-2-1`    | The 2nd `NK1` (next of kin), family name                        |
| `MSH-7`         | Message date/time (`MSH-1` is the field separator, so `MSH` numbering matches the spec) |
| `OBX-5`         | Observation value, free text when `OBX-2` is `TX`/`FT`/`CE`     |

To use your own rules, change a **copy** of the model. `setMappingRules()` replaces the default map, so the copy de-identifies only the paths you give. Here both models target only the family name of the 2nd next of kin:

In [22]:
hl7_two_nk1 = r"""MSH|^~\&|APP|FAC|APP2|FAC2|20240108071200||ADT^A01|1|P|2.5
PID|1||MRN123456||DOE^JOHN||19800101|M
NK1|1|ADAMS^ALICE|SPO
NK1|2|BROWN^BOB|CHD"""

occurrence_rules = {"NK1(2)-2-1": "LAST_NAME"}

base_occurrence = hl7_base.copy({}).setMappingRules(occurrence_rules)
print(base_occurrence.deidentify(hl7_two_nk1))

MSH|^~\&|APP|FAC|APP2|FAC2|20240108071200||ADT^A01|1|P|2.5
PID|1||MRN123456||DOE^JOHN||19800101|M
NK1|1|ADAMS^ALICE|SPO
NK1|2|OLDEN^BOB|CHD


In [23]:
free_text_occurrence = (
    hl7_free_text.copy({})
      .setMappingRules(occurrence_rules)
      .setPipeline(spark, deid_pipeline, "obfuscated")
)
print(free_text_occurrence.deidentify(hl7_two_nk1))

MSH|^~\&|APP|FAC|APP2|FAC2|20240108071200||ADT^A01|1|P|2.5
PID|1||MRN123456||DOE^JOHN||19800101|M
NK1|1|ADAMS^ALICE|SPO
NK1|2|OLDEN^BOB|CHD


## 🧩 Extend the Default PHI Map

Start from the model's default map, add the paths that matter for your feed and set the result on a copy. Here we add the insurance company name (`IN1-4`, an `ORGANIZATION`) to both models:

In [24]:
extended_rules = dict(default_rules)
extended_rules["IN1-4"] = "ORGANIZATION"

base_extended = hl7_base.copy({}).setMappingRules(extended_rules)
print(base_extended.deidentify(hl7_adt))

MSH|^~\&|GHH_ADT|GOOD HEALTH HOSPITAL|GHH_LAB|GOOD HEALTH LAB|20240128071200||ADT^A01|MSG00001|P|2.5
EVN|A01|20240128071200
PID|1||TMC678901^^^GHH^MR||CEASE^MYLENE^ANOLA||19800121|M||W|1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN||162-000-6677|162-000-9911|EN|V||PHH443322|678-90-1234
NK1|1|CEASE^DELTA^FARRELL|SPO|1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN|162-000-6699
PV1|1|I|WARD1^101^A||||559222^LITTER^DERREK^A^^^Dr|553519^PROW^RODERICK^B^^^Dr|||||||||559222^LITTER^DERREK^A^^^Dr||WC678|||||||||||||||||||||||||20240128071200
IN1|1|PLAN01|INS01|Ppl Corporation||||||||||||CEASE^MYLENE^ANOLA|SELF|19800121|1011 North Cooper Street^^Allenchester^VA^57653^USA


In [25]:
free_text_extended = (
    hl7_free_text.copy({})
      .setMappingRules(extended_rules)
      .setPipeline(spark, deid_pipeline, "obfuscated")
)
print(free_text_extended.deidentify(hl7_adt))

MSH|^~\&|GHH_ADT|GOOD HEALTH HOSPITAL|GHH_LAB|GOOD HEALTH LAB|20240128071200||ADT^A01|MSG00001|P|2.5
EVN|A01|20240128071200
PID|1||TMC678901^^^GHH^MR||CEASE^MYLENE^ANOLA||19800121|M||W|1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN||162-000-6677|162-000-9911|EN|V||PHH443322|678-90-1234
NK1|1|CEASE^DELTA^FARRELL|SPO|1011 North Cooper Street^^Allenchester^VA^57653^BHUTAN|162-000-6699
PV1|1|I|WARD1^101^A||||559222^LITTER^DERREK^A^^^Dr|553519^PROW^RODERICK^B^^^Dr|||||||||559222^LITTER^DERREK^A^^^Dr||WC678|||||||||||||||||||||||||20240128071200
IN1|1|PLAN01|INS01|Ppl Corporation||||||||||||CEASE^MYLENE^ANOLA|SELF|19800121|1011 North Cooper Street^^Allenchester^VA^57653^USA


## 🎭 Mask Mode

`setMode("mask")` replaces each value instead of substituting a fake one; `setMaskingPolicy()` picks the style (`same_length_chars`, `entity_labels`, `fixed_length_chars`). Dates are shifted, never masked, so a masking map must not contain date entities. We drop them from the default map. For the free-text model the pipeline's masked text lives in the `masked` column, so we attach the pipeline with that column.

In [26]:
date_entities = ["DATE", "DOB", "DOD"]
mask_rules = {path: entity for path, entity in default_rules.items() if entity not in date_entities}

print(f"Default map: {len(default_rules)} rules, mask map without dates: {len(mask_rules)} rules")

Default map: 280 rules, mask map without dates: 215 rules


In [27]:
base_mask = (
    hl7_base.copy({})
      .setMode("mask")
      .setMaskingPolicy("entity_labels")
      .setMappingRules(mask_rules)
)
print(base_mask.deidentify(hl7_oru))

MSH|^~\&|LAB|GOOD HEALTH LAB|EHR|GOOD HEALTH HOSPITAL|20240108090000||ORU^R01|MSG00002|P|2.4
PID|1||<ID>^^^GHH^MR||<LAST_NAME>^<FIRST_NAME>^<FIRST_NAME>||19800101|M|||<STREET>^^<CITY>^<STATE>^<ZIP>^<COUNTRY>||<PHONE>
OBR|1|ORD123|FILL456|CBC^Complete Blood Count|||20240108083000|||||||||<ID>^<LAST_NAME>^<FIRST_NAME>
OBX|1|TX|NARR^Narrative||Patient John Doe was seen on 01/08/2024 at Good Health Hospital and reports mild fatigue. He lives at 123 Main Street, Boston and can be reached at 617-555-1122.||||||F
NTE|1||Follow up with Dr. Robert Smith at 617-555-9000 in two weeks.


In [28]:
free_text_mask = (
    hl7_free_text.copy({})
      .setMode("mask")
      .setMaskingPolicy("entity_labels")
      .setMappingRules(mask_rules)
      .setPipeline(spark, deid_pipeline, "masked")
)
print(free_text_mask.deidentify(hl7_oru))

MSH|^~\&|LAB|GOOD HEALTH LAB|EHR|GOOD HEALTH HOSPITAL|20240108090000||ORU^R01|MSG00002|P|2.4
PID|1||<ID>^^^GHH^MR||<LAST_NAME>^<FIRST_NAME>^<FIRST_NAME>||19800101|M|||<STREET>^^<CITY>^<STATE>^<ZIP>^<COUNTRY>||<PHONE>
OBR|1|ORD123|FILL456|CBC^Complete Blood Count|||20240108083000|||||||||<ID>^<LAST_NAME>^<FIRST_NAME>
OBX|1|TX|NARR^Narrative||Patient <PATIENT> was seen on <DATE> at <HOSPITAL> and reports mild fatigue. He lives at <STREET>, <CITY> and can be reached at <PHONE>.||||||F
NTE|1||Follow up with Dr. <DOCTOR> at <PHONE> in two weeks.


# Real World Examples

We pull real HL7 v2 sample messages from a public GitHub repository. Any repository or folder of `.hl7` files works, the loader below globs `*.hl7` recursively, so you can point it at your own data by changing the clone URL.

In [29]:
# Clone a public repository that ships HL7 v2 sample messages.
!git clone --depth 1 https://github.com/LinuxForHealth/hl7v2-fhir-converter.git

Cloning into 'hl7v2-fhir-converter'...
remote: Enumerating objects: 379, done.
remote: Counting objects: 100% (379/379), done.
remote: Compressing objects: 100% (318/318), done.
remote: Total 379 (delta 73), reused 151 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (379/379), 440.82 KiB | 2.81 MiB/s, done.
Resolving deltas: 100% (73/73), done.


In [30]:
import glob

hl7_files = sorted(glob.glob("hl7v2-fhir-converter/**/*.hl7", recursive=True))

print(f"Found {len(hl7_files)} HL7 v2 files")
for path in hl7_files:
    print("  -", path)

Found 4 HL7 v2 files
  - hl7v2-fhir-converter/src/test/resources/ORU-multiline-short-mixed.hl7
  - hl7v2-fhir-converter/src/test/resources/ORU-multiline-short.hl7
  - hl7v2-fhir-converter/src/test/resources/sample_unix.hl7
  - hl7v2-fhir-converter/src/test/resources/sample_win.hl7


In [31]:
def read_hl7(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

In [32]:
real_hl7 = read_hl7(hl7_files[0])
print(real_hl7[:600])

MSH|^~\&|WHI_Automation|IBM_Toronto_Lab|IMAGING_REPORT|Hartland|20200802124455||ORU^R01^ORU_R01|MSGID00231|T|2.6||||||||||||^4086::132:2A57:3C28^IPv6
PID|1||0d70c6c8^^^MRN||Patient^Autogenerated||19630306|M||Caucasian|^^^^L6G 1C7~^^^ON~^^Unionville~&Warden Av~8200~^^^^^^B||^^^^^^4042808~^^^^^905~^^^^001|||Married|Baptist|Account_0d70c6c8
NTE|1||Created for MRN: 0d70c6c8
PV1|1|I|^^^Toronto^^^8200 Warden Av|EM|||2905^Langa^Albert^J^IV||0007^SINGH^BALDEV||||||||5755^Kuczma^Sean^^Jr||Visit_0d70c6c8|||||||||||||||||||||||||20200802124455||||||||ABC
OBR|1|PON_0d70c6c8^LAB|FON_0d70c6c8^LAB|1487^ECHO 


In [33]:
real_base = hl7_base.deidentify(real_hl7)
real_free_text = hl7_free_text.deidentify(real_hl7)

print("hl7v2_deidentification_base")
show_hl7_diff(real_hl7, real_base)

hl7v2_deidentification_base
Changed: MSH-7 | 20200802124455 → 20200822124455
Changed: PID-3 | 0d70c6c8^^^MRN → 5q25h1h3^^^MRN
Changed: PID-5 | Patient^Autogenerated → Lustig^Conny
Changed: PID-7 | 19630306 → 19630326
Changed: PID-11 | ^^^^L6G 1C7~^^^ON~^^Unionville~&Warden Av~8200~^^^^^^B → ^^^^K1R 6H2~^^^KS~^^Lake Shelbyshire~1000 West Carson Street~1912 HAYES AVENUE~^^^^^^B
Changed: PID-13 | ^^^^^^4042808~^^^^^905~^^^^001 → 9597353~450~556
Changed: PID-16 | Married → Tpmmjzq
Changed: PID-17 | Baptist → Ypuejve
Changed: PID-18 | Account_0d70c6c8 → Phhlnce_5q25h1h3
Changed: PV1-7 | 2905^Langa^Albert^J^IV → 7450^Merry^Steward^J^IV
Changed: PV1-9 | 0007^SINGH^BALDEV → 5552^ASAI^NATHEN
Changed: PV1-17 | 5755^Kuczma^Sean^^Jr → 0200^Skeeter^Tedra^^Jr
Changed: PV1-19 | Visit_0d70c6c8 → Wjvje_5q25h1h3
Changed: PV1-44 | 20200802124455 → 20200822124455
Changed: PV1-52 | ABC → PYH
Changed: OBR-6 | 20200802124455 → 20200822124455
Changed: OBR-7 | 20200802124455 → 20200822124455
Changed: OBR-16 | 

In [34]:
print("hl7v2_deidentification_free_text")
show_hl7_diff(real_hl7, real_free_text)

hl7v2_deidentification_free_text
Changed: MSH-7 | 20200802124455 → 20200822124455
Changed: PID-3 | 0d70c6c8^^^MRN → 5q25h1h3^^^MRN
Changed: PID-5 | Patient^Autogenerated → Lustig^Conny
Changed: PID-7 | 19630306 → 19630326
Changed: PID-11 | ^^^^L6G 1C7~^^^ON~^^Unionville~&Warden Av~8200~^^^^^^B → ^^^^K1R 6H2~^^^KS~^^Lake Shelbyshire~1000 West Carson Street~1912 HAYES AVENUE~^^^^^^B
Changed: PID-13 | ^^^^^^4042808~^^^^^905~^^^^001 → 9597353~450~556
Changed: PID-16 | Married → Tpmmjzq
Changed: PID-17 | Baptist → Ypuejve
Changed: PID-18 | Account_0d70c6c8 → Phhlnce_5q25h1h3
Changed: NTE-3 | Created for MRN: 0d70c6c8 → Created for MRN: 2i92z8z0
Changed: PV1-7 | 2905^Langa^Albert^J^IV → 7450^Merry^Steward^J^IV
Changed: PV1-9 | 0007^SINGH^BALDEV → 5552^ASAI^NATHEN
Changed: PV1-17 | 5755^Kuczma^Sean^^Jr → 0200^Skeeter^Tedra^^Jr
Changed: PV1-19 | Visit_0d70c6c8 → Wjvje_5q25h1h3
Changed: PV1-44 | 20200802124455 → 20200822124455
Changed: PV1-52 | ABC → PYH
Changed: OBR-6 | 20200802124455 → 202008

## All Files at Once

In [35]:
all_hl7_contents = [read_hl7(path) for path in hl7_files]

all_base = hl7_base.deidentify_list(all_hl7_contents)
all_free_text = [hl7_free_text.deidentify(content) for content in all_hl7_contents]

for path, original, out_base, out_free_text in zip(hl7_files, all_hl7_contents, all_base, all_free_text):
    print("=" * 100)
    print(os.path.basename(path))
    print("Same output from both models:", out_base == out_free_text)

ORU-multiline-short-mixed.hl7
Same output from both models: False
ORU-multiline-short.hl7
Same output from both models: False
sample_unix.hl7
Same output from both models: True
sample_win.hl7
Same output from both models: True


Look at the second file with each model:

In [36]:
print("hl7v2_deidentification_base")
show_hl7_diff(all_hl7_contents[1], all_base[1])

hl7v2_deidentification_base
Changed: MSH-7 | 20200802124455 → 20200822124455
Changed: PID-3 | 0d70c6c8^^^MRN → 5q25h1h3^^^MRN
Changed: PID-5 | Patient^Autogenerated → Lustig^Conny
Changed: PID-7 | 19630306 → 19630326
Changed: PID-11 | ^^^^L6G 1C7~^^^ON~^^Unionville~&Warden Av~8200~^^^^^^B → ^^^^K1R 6H2~^^^KS~^^Lake Shelbyshire~1000 West Carson Street~1912 HAYES AVENUE~^^^^^^B
Changed: PID-13 | ^^^^^^4042808~^^^^^905~^^^^001 → 9597353~450~556
Changed: PID-16 | Married → Tpmmjzq
Changed: PID-17 | Baptist → Ypuejve
Changed: PID-18 | Account_0d70c6c8 → Phhlnce_5q25h1h3
Changed: PV1-7 | 2905^Langa^Albert^J^IV → 7450^Merry^Steward^J^IV
Changed: PV1-9 | 0007^SINGH^BALDEV → 5552^ASAI^NATHEN
Changed: PV1-17 | 5755^Kuczma^Sean^^Jr → 0200^Skeeter^Tedra^^Jr
Changed: PV1-19 | Visit_0d70c6c8 → Wjvje_5q25h1h3
Changed: PV1-44 | 20200802124455 → 20200822124455
Changed: PV1-52 | ABC → PYH
Changed: OBR-6 | 20200802124455 → 20200822124455
Changed: OBR-7 | 20200802124455 → 20200822124455
Changed: OBR-16 | 

In [37]:
print("hl7v2_deidentification_free_text")
show_hl7_diff(all_hl7_contents[1], all_free_text[1])

hl7v2_deidentification_free_text
Changed: MSH-7 | 20200802124455 → 20200822124455
Changed: PID-3 | 0d70c6c8^^^MRN → 5q25h1h3^^^MRN
Changed: PID-5 | Patient^Autogenerated → Lustig^Conny
Changed: PID-7 | 19630306 → 19630326
Changed: PID-11 | ^^^^L6G 1C7~^^^ON~^^Unionville~&Warden Av~8200~^^^^^^B → ^^^^K1R 6H2~^^^KS~^^Lake Shelbyshire~1000 West Carson Street~1912 HAYES AVENUE~^^^^^^B
Changed: PID-13 | ^^^^^^4042808~^^^^^905~^^^^001 → 9597353~450~556
Changed: PID-16 | Married → Tpmmjzq
Changed: PID-17 | Baptist → Ypuejve
Changed: PID-18 | Account_0d70c6c8 → Phhlnce_5q25h1h3
Changed: NTE-3 | Created for MRN: 0d70c6c8 → Created for MRN: 2i92z8z0
Changed: PV1-7 | 2905^Langa^Albert^J^IV → 7450^Merry^Steward^J^IV
Changed: PV1-9 | 0007^SINGH^BALDEV → 5552^ASAI^NATHEN
Changed: PV1-17 | 5755^Kuczma^Sean^^Jr → 0200^Skeeter^Tedra^^Jr
Changed: PV1-19 | Visit_0d70c6c8 → Wjvje_5q25h1h3
Changed: PV1-44 | 20200802124455 → 20200822124455
Changed: PV1-52 | ABC → PYH
Changed: OBR-6 | 20200802124455 → 202008

Pick a file and a model in the widget below, de-identify it and save the result next to the input.

In [38]:
from ipywidgets import Dropdown, Button, Output, VBox, HTML
from IPython.display import display, clear_output

models = {"hl7v2_deidentification_base": hl7_base, "hl7v2_deidentification_free_text": hl7_free_text}

file_dropdown = Dropdown(options={os.path.basename(p): p for p in hl7_files}, description="HL7 File:",
                         style={"description_width": "initial"}, layout={"width": "600px"})
model_dropdown = Dropdown(options=list(models), description="Model:",
                          style={"description_width": "initial"}, layout={"width": "600px"})
output = Output()

def deidentify_selected(b):
    with output:
        clear_output()
        selected_path = file_dropdown.value
        hl7_content = read_hl7(selected_path)
        deidentified_content = models[model_dropdown.value].deidentify(hl7_content)

        print("Processing:", os.path.basename(selected_path), "with", model_dropdown.value)
        print("=" * 80)
        show_hl7_diff(hl7_content, deidentified_content)

        out_path = selected_path.replace(".hl7", "_deidentified.hl7")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(deidentified_content)
        print("\nSaved to:", out_path)

button = Button(description="De-identify Selected HL7", button_style="success", layout={"width": "320px"})
button.on_click(deidentify_selected)

display(HTML("<h3>Select an HL7 v2 file and a model:</h3>"))
display(VBox([file_dropdown, model_dropdown, button, output]))

HTML(value='<h3>Select an HL7 v2 file and a model:</h3>')